# Potassium (Hyperkalemia) from the 12-lead ECG — MIMIC-IV-ECG + MIMIC-IV (Kaggle)

**Before Run All**
1. Settings → Internet **On** · keep this notebook **Private**. Accelerator: **None** while waveforms download
   (hours, no GPU needed — saves your GPU quota), then **GPU T4 x2** for training.
2. Add-ons → Google Cloud Services (BigQuery + Cloud Storage) or Google Cloud SDK — same as the other projects.
3. Check `project_id` in the config cell. The private bucket `<project_id>-mimic-ckpt` is reused/created automatically.

**What happens (everything resumable — if the session dies, just Run All again with the same `run_name`):**
1. Labels: every ECG with a potassium result within ±60 min (BigQuery, one query, cached).
2. Waveforms: all labelled ECGs are downloaded from the open MIMIC-IV-ECG mirror, resampled to 250 Hz and written
   as float16 shards to your bucket (finished shards are skipped on re-run).
3. Training: 1D ResNet on 2×T4 with mixed precision; full state checkpointed every 10 min + every epoch;
   stops cleanly at 11.3 h.
4. Test evaluation, machine-measurement baseline, population Grad-CAM, `ecg_k_resnet1d.pth`, GitHub report.

## 1 · Setup

In [ ]:
!pip install -q wfdb google-cloud-bigquery-storage db-dtypes 2>&1 | tail -1

## 2 · Project code
These cells write the package to `/kaggle/working/repo`. No need to edit them; the steps further down call this code.

In [ ]:
!mkdir -p /kaggle/working/repo/src

In [ ]:
%%writefile /kaggle/working/repo/src/__init__.py
"""package"""


In [ ]:
%%writefile /kaggle/working/repo/src/config.py
from __future__ import annotations

from dataclasses import asdict, dataclass, field


@dataclass
class Config:
    # --- GCP (only project_id is required) ---
    project_id: str = ""
    bucket: str = ""                     # empty = auto-create private "<project_id>-mimic-ckpt"
    bucket_location: str = "US"
    gcs_prefix: str = "ecg-potassium"
    run_name: str = "main"               # same name = resume
    local_store: str = ""                # tests only

    # --- data sources ---
    hosp_dataset: str = ""               # empty = auto-detect (potassium labs are credentialed MIMIC-IV)
    hosp_candidates: list = field(default_factory=lambda: ["mimiciv_v3_1_hosp", "mimiciv_hosp", "mimiciv_3_1_hosp", "mimiciv_v2_2_hosp"])
    # MIMIC-IV-ECG is open access: public AWS mirror first, PhysioNet as fallback (no login needed)
    wave_base_urls: list = field(default_factory=lambda: [
        "https://physionet-open.s3.amazonaws.com/mimic-iv-ecg/1.0/",
        "https://physionet-open.s3.us-east-1.amazonaws.com/mimic-iv-ecg/1.0/",
        "https://s3.amazonaws.com/physionet-open/mimic-iv-ecg/1.0/",
        "https://physionet.org/files/mimic-iv-ecg/1.0/"])
    s3_bucket_url: str = "https://physionet-open.s3.amazonaws.com/"   # public AWS Open Data bucket (auto-discovers the prefix)
    s3_dataset: str = "mimic-iv-ecg"
    wave_source: str = "zip"             # zip = PhysioNet ZIP via HTTP Range (fast, few connections) | files | auto
    zip_url: str = ""                    # empty = read the link from the PhysioNet project page
    zip_chunk_mb: float = 256            # one checkpointed pack per ~256 MB of the ZIP
    zip_parallel: int = 6                # parallel range downloads (physionet.org gives ~0.3 MB/s per connection)
    physionet_org_workers: int = 4       # physionet.org throttles heavy parallel downloads
    max_fail_frac: float = 0.02          # a shard with more failures is NOT committed (retried next run)

    # --- labels ---
    k_itemids: tuple = (50971, 50822)    # serum potassium (chemistry) + whole-blood potassium (blood gas)
    window_min: int = 60                 # potassium drawn within +-60 min of the ECG
    exclude_hemolyzed: bool = True       # hemolysis falsely raises K+
    k_min: float = 1.5
    k_max: float = 10.0
    hyper_thr: float = 5.5
    severe_thr: float = 6.0
    hypo_thr: float = 3.5
    max_ecgs: int | None = None          # None = FULL dataset (every ECG with a matched K+)
    split: tuple = (0.8, 0.1, 0.1)       # patient-level train/val/test
    seed: int = 42

    # --- waveform preprocessing (done once, sharded into the bucket, resumable) ---
    fs_out: int = 250                    # 500 Hz -> 250 Hz
    shard_size: int = 4096               # ECGs per shard (~245 MB float16)
    download_workers: int = 64
    cache_dir: str = "/tmp/ecgk_cache"   # local shard cache (auto-evicted when disk is low)
    min_free_gb: float = 8.0

    # --- model / training ---
    width: int = 64
    kernel: int = 7
    dropout: float = 0.3
    epochs: int = 25
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-2
    warmup_frac: float = 0.03
    patience: int = 5                    # early stopping on val AUROC (K >= hyper_thr)
    amp: bool = True
    ckpt_minutes: float = 10.0           # full training state -> bucket at least this often
    max_runtime_hours: float = 11.3      # stop cleanly before Kaggle's 12 h limit

    # --- evaluation ---
    n_boot: int = 500
    gradcam_n: int = 2000                # ECGs per group used for the population Grad-CAM beat
    min_cell: int = 10

    # --- paths ---
    work_dir: str = "/kaggle/working/ecg_potassium"     # model + aggregate report
    private_dir: str = "/tmp/ecgk_private"              # labels / predictions (patient-level)
    cleanup_private: bool = True

    train_needs_gpu: bool = True         # CPU session: stop after downloading shards (no slow CPU training)

    # --- tests ---
    debug_crash_after_shards: int | None = None

    def to_dict(self):
        return asdict(self)


In [ ]:
%%writefile /kaggle/working/repo/src/data.py
"""Shard loader: local disk cache (auto-evicted when space is low) + background prefetch from the store."""
from __future__ import annotations

import logging
import queue
import shutil
import threading
from pathlib import Path

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)


class ShardSet:
    def __init__(self, cfg, store, entries):
        self.cfg, self.store = cfg, store
        self.entries = [(k, n) for k, n in entries if n > 0]
        self.cache = Path(cfg.cache_dir)
        self.cache.mkdir(parents=True, exist_ok=True)
        self.n = sum(n for _, n in self.entries)

    def _free_gb(self):
        return shutil.disk_usage(self.cache).free / 1e9

    def _evict(self, keep):
        files = sorted(self.cache.rglob("*.npy"), key=lambda p: p.stat().st_atime)
        for f in files:
            if self._free_gb() > self.cfg.min_free_gb:
                break
            if f.with_suffix("").as_posix().endswith(keep):
                continue
            f.unlink(missing_ok=True)
            f.with_suffix(".parquet").unlink(missing_ok=True)

    def load(self, key):
        x, m = self.cache / f"{key}.npy", self.cache / f"{key}.parquet"
        if not (x.exists() and m.exists()):
            self._evict(keep=key)
            self.store.get(f"{key}.parquet", m)
            self.store.get(f"{key}.npy", x)
        return np.load(x, mmap_mode="r"), pd.read_parquet(m)

    def iterate(self, order, prefetch=2):
        """Yield (idx, key, X, meta) in the given shard order, loading the next shards in background."""
        q: queue.Queue = queue.Queue(maxsize=prefetch)

        def producer():
            try:
                for i in order:
                    k = self.entries[i][0]
                    X, m = self.load(k)
                    q.put((i, k, np.ascontiguousarray(X), m))
            except Exception as e:      # surface errors in the consumer
                q.put(e)
            q.put(None)
        threading.Thread(target=producer, daemon=True).start()
        while True:
            item = q.get()
            if item is None:
                return
            if isinstance(item, Exception):
                raise item
            yield item


In [ ]:
%%writefile /kaggle/working/repo/src/evaluate.py
"""Test metrics with bootstrap CIs, subgroups, machine-measurement baseline, population Grad-CAM beat."""
from __future__ import annotations

import logging

import numpy as np
import pandas as pd
import torch
from scipy.signal import find_peaks
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve

from .model import gradcam
from .train import device

log = logging.getLogger(__name__)


def _boot(y, p, fn, n, seed):
    y, p = np.asarray(y), np.asarray(p)
    if not 0 < y.sum() < len(y):
        return np.nan, np.nan, np.nan
    rng, v = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            v.append(fn(y[i], p[i]))
    return float(fn(y, p)), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


def classification(pred: pd.DataFrame, cfg) -> pd.DataFrame:
    rows = []
    for name, y, p in ((f"K ≥ {cfg.hyper_thr}", pred.k >= cfg.hyper_thr, pred.p_hyper),
                       (f"K ≥ {cfg.severe_thr}", pred.k >= cfg.severe_thr, pred.p_hyper),
                       (f"K < {cfg.hypo_thr}", pred.k < cfg.hypo_thr, pred.p_hypo)):
        y = y.astype(int).to_numpy()
        auc = _boot(y, p, roc_auc_score, cfg.n_boot, cfg.seed)
        ap = _boot(y, p, average_precision_score, cfg.n_boot, cfg.seed)
        sens90 = np.nan
        if 0 < y.sum() < len(y):
            fpr, tpr, _ = roc_curve(y, p)
            sens90 = float(tpr[np.searchsorted(fpr, 0.10, side="right") - 1])   # sensitivity at 90% specificity
        rows.append({"Target": name, "n positive": int(y.sum()), "prevalence": y.mean(),
                     "AUROC": auc[0], "AUROC lo": auc[1], "AUROC hi": auc[2],
                     "AUPRC": ap[0], "Sens @ 90% spec": sens90})
    return pd.DataFrame(rows)


def regression(pred: pd.DataFrame) -> dict:
    e = pred.pred_k - pred.k
    return dict(mae=float(e.abs().mean()), rmse=float(np.sqrt((e ** 2).mean())),
                pearson_r=float(np.corrcoef(pred.pred_k, pred.k)[0, 1]),
                within_0_5=float((e.abs() <= 0.5).mean()))


def subgroups(pred: pd.DataFrame, cfg) -> pd.DataFrame:
    p = pred.copy()
    p["age_group"] = pd.cut(p.age, [0, 39, 64, 79, 200], labels=["18-39", "40-64", "65-79", "80+"]).astype(str)
    p["sex"] = p.gender.fillna("Unknown")
    rows = []
    for a in ("sex", "age_group"):
        for g, d in p.groupby(a):
            y = (d.k >= cfg.hyper_thr).astype(int)
            ok = y.sum() >= cfg.min_cell and (len(y) - y.sum()) >= cfg.min_cell
            rows.append(dict(attribute=a, group=g, n=len(d) if len(d) >= cfg.min_cell else f"<{cfg.min_cell}",
                             n_positive=int(y.sum()) if y.sum() >= cfg.min_cell else f"<{cfg.min_cell}",
                             auroc=float(roc_auc_score(y, d.p_hyper)) if ok else np.nan,
                             mae=float((d.pred_k - d.k).abs().mean()) if len(d) >= cfg.min_cell else np.nan))
    return pd.DataFrame(rows)


MM_COLS = ["rr_interval", "p_onset", "p_end", "qrs_onset", "qrs_end", "t_end", "p_axis", "qrs_axis", "t_axis"]


def mm_features(mm: pd.DataFrame) -> pd.DataFrame:
    f = mm[["study_id"] + [c for c in MM_COLS if c in mm]].copy()
    for c in MM_COLS:
        if c in f:
            f[c] = pd.to_numeric(f[c], errors="coerce")
            f.loc[f[c].abs() >= 29999, c] = np.nan            # machine "not measured" sentinel
    f["hr"] = 60000 / f["rr_interval"]
    f["qrs_dur"] = f["qrs_end"] - f["qrs_onset"]
    f["pr"] = f["qrs_onset"] - f["p_onset"]
    f["qt"] = f["t_end"] - f["qrs_onset"]
    f["qtc"] = f["qt"] / np.sqrt(f["rr_interval"] / 1000)
    return f


def baseline_machine(labels: pd.DataFrame, mm: pd.DataFrame, cfg) -> dict:
    """XGBoost on the ECG machine's own interval/axis measurements (the classical approach)."""
    import xgboost as xgb
    f = mm_features(mm)
    d = labels.merge(f, on="study_id", how="inner")
    cols = [c for c in f.columns if c != "study_id"]
    tr, te = d[d.split == "train"], d[d.split == "test"]
    if len(tr) < 100 or len(te) < 50:
        return {}
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          tree_method="hist", device=dev, eval_metric="auc")
    m.fit(tr[cols], (tr.k >= cfg.hyper_thr).astype(int))
    p = m.predict_proba(te[cols])[:, 1]
    y = (te.k >= cfg.hyper_thr).astype(int)
    auc = _boot(y, p, roc_auc_score, cfg.n_boot, cfg.seed)
    return dict(auroc=auc[0], auroc_lo=auc[1], auroc_hi=auc[2], auprc=float(average_precision_score(y, p)),
                n_test=int(len(te)), features=cols)


def gradcam_beats(model, shardset, pred: pd.DataFrame, cfg, fs=None) -> dict:
    """Average lead-II beat and Grad-CAM (hyperkalemia logit) aligned on R-peaks, per group (aggregate only)."""
    fs = fs or cfg.fs_out
    dev = device()
    pre, post = int(0.25 * fs), int(0.60 * fs)
    want = {"hyper (true K ≥ thr)": pred.k >= cfg.hyper_thr, "normal (3.5–5.0)": pred.k.between(3.5, 5.0)}
    ids = {g: set(pred.loc[m, "study_id"].sample(min(cfg.gradcam_n, int(m.sum())), random_state=cfg.seed))
           for g, m in want.items()}
    acc = {g: {"ecg": [], "cam": []} for g in want}
    for _, _, X, meta in shardset.iterate(range(len(shardset.entries))):
        for g in want:
            idx = np.where(meta.study_id.isin(ids[g]).to_numpy())[0]
            for b in range(0, len(idx), 128):
                j = idx[b:b + 128]
                xb = torch.from_numpy(np.asarray(X[j], dtype=np.float32)).to(dev)
                cam = gradcam(model, xb).cpu().numpy()
                lead2 = xb[:, 1].cpu().numpy()
                for s, c in zip(lead2, cam):
                    a = np.abs(s)
                    pk, _ = find_peaks(a, distance=int(0.3 * fs), height=0.5 * np.percentile(a, 99))
                    for r in pk:
                        if r - pre >= 0 and r + post <= len(s):
                            acc[g]["ecg"].append(s[r - pre:r + post] * np.sign(s[r]))   # R upright
                            acc[g]["cam"].append(c[r - pre:r + post] * len(s))
    t = (np.arange(-pre, post) / fs) * 1000
    res = {"t_ms": t.tolist()}
    for g, a in acc.items():
        if len(a["ecg"]) < cfg.min_cell:
            continue
        e, c = np.array(a["ecg"]), np.array(a["cam"])
        qrs = (t >= -50) & (t <= 80)
        stt = (t >= 120) & (t <= 450)
        res[g] = dict(n_beats=len(e), ecg=np.median(e, 0).tolist(), cam=c.mean(0).tolist(),
                      cam_share_qrs=float(c[:, qrs].sum() / c.sum()), cam_share_st_t=float(c[:, stt].sum() / c.sum()))
    return res


def plots(pred, cls, beats, hist, base, cfg, fdir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    # ROC
    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    for name, y, p in ((f"K≥{cfg.hyper_thr}", pred.k >= cfg.hyper_thr, pred.p_hyper),
                       (f"K≥{cfg.severe_thr}", pred.k >= cfg.severe_thr, pred.p_hyper),
                       (f"K<{cfg.hypo_thr}", pred.k < cfg.hypo_thr, pred.p_hypo)):
        if 0 < y.sum() < len(y):
            fpr, tpr, _ = roc_curve(y, p)
            ax.plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=8)
    ax.set_title("ECG-only model, test set"); fig.tight_layout(); fig.savefig(fdir / "roc.png", dpi=150); plt.close(fig)
    # predicted vs true (2D histogram = aggregate, no individual points)
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    h = ax.hist2d(pred.k, pred.pred_k, bins=[np.arange(2, 8.01, .1), np.arange(2, 8.01, .1)], cmin=cfg.min_cell, cmap="viridis")
    fig.colorbar(h[3], ax=ax, label=f"ECGs per bin (bins <{cfg.min_cell} hidden)")
    ax.plot([2, 8], [2, 8], "w--", lw=.8); ax.set_xlabel("measured K+ (mmol/L)"); ax.set_ylabel("ECG-predicted K+")
    fig.tight_layout(); fig.savefig(fdir / "pred_vs_true.png", dpi=150); plt.close(fig)
    # Grad-CAM beat
    groups = [g for g in beats if g != "t_ms"]
    if groups:
        t = np.array(beats["t_ms"])
        fig, axes = plt.subplots(1, len(groups), figsize=(5.2 * len(groups), 3.6), squeeze=False)
        for ax, g in zip(axes[0], groups):
            b = beats[g]
            ax.plot(t, b["ecg"], color="k", lw=1.2, label="median beat (lead II)")
            ax2 = ax.twinx()
            ax2.fill_between(t, 0, b["cam"], color="tab:red", alpha=.35, label="Grad-CAM")
            ax.axvspan(120, 450, color="tab:blue", alpha=.07)
            ax.set_title(f"{g}\nST–T share {b['cam_share_st_t']:.0%} · QRS share {b['cam_share_qrs']:.0%}", fontsize=9)
            ax.set_xlabel("ms from R peak"); ax2.set_yticks([])
        fig.tight_layout(); fig.savefig(fdir / "gradcam_beat.png", dpi=150); plt.close(fig)
    # training curve
    if hist:
        h = pd.DataFrame(hist)
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ax.plot(h.epoch, h.val_auroc_hyper, "o-", label="val AUROC")
        ax2 = ax.twinx(); ax2.plot(h.epoch, h.val_mae, "s--", color="tab:orange", label="val MAE")
        ax.set_xlabel("epoch"); ax.set_ylabel("AUROC"); ax2.set_ylabel("MAE")
        fig.tight_layout(); fig.savefig(fdir / "training.png", dpi=150); plt.close(fig)


In [ ]:
%%writefile /kaggle/working/repo/src/gcp.py
"""BigQuery helpers. Works with Kaggle's "Google Cloud Services" add-on
(linked account) and with normal ADC credentials elsewhere."""
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

log = logging.getLogger(__name__)
SOURCE_PROJECT = "physionet-data"


def _kaggle_integrations() -> str:
    return os.environ.get("KAGGLE_KERNEL_INTEGRATIONS", "")


def credentials(target: str):
    """Credentials for 'bigquery' or 'gcs'.
    On Kaggle: the account attached in Add-ons -> Google Cloud Services (passed explicitly, so we never
    fall back to the GCE metadata server). Elsewhere: Application Default Credentials (None)."""
    if "KAGGLE_USER_SECRETS_TOKEN" not in os.environ:
        return None                                   # not on Kaggle -> ADC
    want = {"bigquery": "BIGQUERY", "gcs": "GCS"}[target]
    have = [x.upper() for x in _kaggle_integrations().split(":") if x]
    if want in have:
        from kaggle_secrets import GcpTarget
        try:
            from kaggle_gcp import KaggleKernelCredentials
        except ImportError:                           # minimal re-implementation
            from google.auth import credentials as gac
            from kaggle_secrets import UserSecretsClient

            class KaggleKernelCredentials(gac.Credentials):
                def __init__(self, target):
                    super().__init__()
                    self.target = target

                def refresh(self, request):
                    self.token, self.expiry = UserSecretsClient()._get_access_token(self.target)
        return KaggleKernelCredentials(target=GcpTarget[want])
    try:                                              # Add-ons -> "Google Cloud SDK" also works
        from google.oauth2.credentials import Credentials
        from kaggle_secrets import UserSecretsClient
        info = json.loads(UserSecretsClient().get_gcloud_credential())
        log.info("Using Google Cloud SDK add-on credential for %s", target)
        return Credentials.from_authorized_user_info(info)
    except Exception:
        pass
    raise RuntimeError(
        f"Kaggle has no {want} integration attached (KAGGLE_KERNEL_INTEGRATIONS='{_kaggle_integrations()}').\n"
        "Fix: Add-ons -> Google Cloud Services -> attach your Google account and tick BOTH 'BigQuery' and "
        "'Cloud Storage' -> Save -> then Run -> Restart & clear outputs -> Run All.")


def validate_project_id(project_id: str) -> str:
    pid = (project_id or "").strip()
    if not pid or "YOUR_" in pid.upper() or " " in pid:
        raise ValueError(
            f"cfg.project_id='{project_id}' is not a real GCP project ID.\n"
            "Find it: console.cloud.google.com -> project picker (top bar) -> copy the 'ID' column "
            "(e.g. 'mimic-research-123456', NOT the display name). Use the same Google account/project "
            "that is attached in Kaggle Add-ons and has billing/BigQuery enabled.")
    return pid


def check_access(project_id: str, probes=()):
    """Quick pre-flight check printed in the notebook."""
    try:
        validate_project_id(project_id)
        print("project_id:", project_id, "OK")
    except ValueError as e:
        print("project_id: FAILED ->", e)
        return
    print("Kaggle integrations:", _kaggle_integrations() or "(none)")
    for t in ("bigquery", "gcs"):
        try:
            c = credentials(t)
            if c is not None:
                from google.auth.transport.requests import Request
                c.refresh(Request())
            print(f"  {t:8s}: OK")
        except Exception as e:
            print(f"  {t:8s}: FAILED -> {e}")
            return
    try:
        client = bq_client(project_id)
        from google.cloud import bigquery
        client.query("SELECT 1", job_config=bigquery.QueryJobConfig(dry_run=True))
        print("  BigQuery project/billing: OK")
    except Exception as e:
        print("  BigQuery project/billing: FAILED ->", str(e)[:300])
        return
    for t in probes:
        ok = _dry_run_ok(client, f"SELECT 1 FROM `{SOURCE_PROJECT}.{t}` LIMIT 1")
        print(f"  PhysioNet {t}: {'OK' if ok else 'not readable (other version name is tried automatically)'}")


def bq_client(project_id: str):
    from google.cloud import bigquery
    project_id = validate_project_id(project_id)
    return bigquery.Client(project=project_id, credentials=credentials("bigquery"))


def _dry_run_ok(client, sql: str) -> bool:
    from google.api_core import exceptions as gexc
    from google.cloud import bigquery
    try:
        client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        return True
    except (gexc.NotFound, gexc.Forbidden, gexc.BadRequest):
        return False            # dataset absent or no PhysioNet access -> try next candidate


def resolve_dataset(client, override: str, candidates: list[str], probe_table: str) -> str:
    """Return 'physionet-data.<dataset>' that the account can read (free dry-run probe)."""
    names = [override] if override else candidates
    for name in names:
        fq = name if "." in name else f"{SOURCE_PROJECT}.{name}"
        if _dry_run_ok(client, f"SELECT 1 FROM `{fq}.{probe_table}` LIMIT 1"):
            log.info("Using dataset %s", fq)
            return fq
    raise RuntimeError(
        f"None of {names} is readable (probe table '{probe_table}'). Open BigQuery console -> "
        f"'{SOURCE_PROJECT}' and set the dataset name in the config.")


def query_df(client, sql: str):
    job = client.query(sql)
    job.result()
    log.info("BigQuery processed %.2f GB", (job.total_bytes_processed or 0) / 1e9)
    try:
        return job.to_dataframe(create_bqstorage_client=True)
    except Exception as e:  # Storage Read API not permitted -> REST fallback
        log.warning("BQ Storage API unavailable (%s). Falling back to REST download.", type(e).__name__)
        return job.result().to_dataframe(create_bqstorage_client=False)


In [ ]:
%%writefile /kaggle/working/repo/src/labels.py
"""ECG list (open MIMIC-IV-ECG) + potassium labels (credentialed MIMIC-IV labevents via BigQuery)."""
from __future__ import annotations

import io
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from . import gcp

log = logging.getLogger(__name__)

K_SQL = """
SELECT subject_id, charttime, itemid, valuenum,
       REGEXP_CONTAINS(LOWER(IFNULL(comments, '')), r'hemoly') AS hemolyzed
FROM `{HOSP}.labevents`
WHERE itemid IN ({ITEMS}) AND valuenum IS NOT NULL AND valuenum BETWEEN {KMIN} AND {KMAX}
"""
PAT_SQL = "SELECT subject_id, gender, anchor_age, anchor_year FROM `{HOSP}.patients`"


def _cached(cfg, store, key, build):
    loc = Path(cfg.private_dir) / key
    if loc.exists() or store.get(key, loc):
        return pd.read_parquet(loc)
    df = build()
    loc.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(loc, index=False)
    store.put(loc, key)
    return df


def _http_csv(cfg, name: str) -> pd.DataFrame:
    import requests
    last = None
    for base in cfg.wave_base_urls:
        try:
            r = requests.get(base + name, timeout=300)
            r.raise_for_status()
            log.info("Downloaded %s from %s (%.0f MB)", name, base, len(r.content) / 1e6)
            return pd.read_csv(io.BytesIO(r.content), low_memory=False)
        except Exception as e:
            last = e
            log.warning("%s from %s failed: %s", name, base, e)
    raise RuntimeError(f"Could not download {name}: {last}")


def record_list(cfg, store) -> pd.DataFrame:
    return _cached(cfg, store, "cache/record_list.parquet", lambda: _http_csv(cfg, "record_list.csv"))


def machine_measurements(cfg, store) -> pd.DataFrame:
    return _cached(cfg, store, "cache/machine_measurements.parquet",
                   lambda: _http_csv(cfg, "machine_measurements.csv"))


def _hosp(cfg):
    client = gcp.bq_client(cfg.project_id)
    return client, gcp.resolve_dataset(client, cfg.hosp_dataset, cfg.hosp_candidates, "labevents")


def potassium(cfg, store) -> pd.DataFrame:
    def build():
        client, hosp = _hosp(cfg)
        return gcp.query_df(client, K_SQL.format(HOSP=hosp, ITEMS=", ".join(map(str, cfg.k_itemids)),
                                                 KMIN=cfg.k_min, KMAX=cfg.k_max))
    return _cached(cfg, store, "cache/potassium.parquet", build)


def patients(cfg, store) -> pd.DataFrame:
    def build():
        client, hosp = _hosp(cfg)
        return gcp.query_df(client, PAT_SQL.format(HOSP=hosp))
    return _cached(cfg, store, "cache/patients.parquet", build)


def match(ecg: pd.DataFrame, k: pd.DataFrame, pat: pd.DataFrame, cfg) -> pd.DataFrame:
    """Nearest (non-hemolyzed) potassium within +-window of each ECG; patient-level split."""
    # BigQuery gives nullable Int64 / datetime64[us]; CSVs give int64 / datetime64[ns] -> unify
    ecg, k, pat = ecg.copy(), k.copy(), pat.copy()
    for d in (ecg, k, pat):
        d.dropna(subset=["subject_id"], inplace=True)
        d["subject_id"] = d["subject_id"].astype("int64")
    ecg["ecg_time"] = pd.to_datetime(ecg["ecg_time"], errors="coerce").dt.tz_localize(None).astype("datetime64[ns]")
    k["charttime"] = pd.to_datetime(k["charttime"], errors="coerce").dt.tz_localize(None).astype("datetime64[ns]")
    k["valuenum"] = pd.to_numeric(k["valuenum"], errors="coerce").astype("float64")
    if "study_id" in ecg:
        ecg["study_id"] = ecg["study_id"].astype("int64")
    if cfg.exclude_hemolyzed and "hemolyzed" in k:
        n0 = len(k)
        k = k[~k["hemolyzed"].astype(bool)]
        log.info("Removed %d hemolyzed K+ results", n0 - len(k))
    ecg = ecg.dropna(subset=["ecg_time"]).sort_values("ecg_time")
    k = k.dropna(subset=["charttime"]).sort_values("charttime")
    m = pd.merge_asof(ecg, k[["subject_id", "charttime", "valuenum", "itemid"]].rename(columns={"valuenum": "k"}),
                      left_on="ecg_time", right_on="charttime", by="subject_id", direction="nearest",
                      tolerance=pd.Timedelta(minutes=cfg.window_min))
    m = m.dropna(subset=["k"]).copy()
    m["k_dt_min"] = (m["charttime"] - m["ecg_time"]).dt.total_seconds() / 60
    m = m.merge(pat, on="subject_id", how="left")
    m["age"] = pd.to_numeric(m["anchor_age"], errors="coerce") + (m["ecg_time"].dt.year - pd.to_numeric(m["anchor_year"], errors="coerce"))
    rng = np.random.default_rng(cfg.seed)
    subs = m["subject_id"].unique()
    u = pd.Series(rng.random(len(subs)), index=subs)
    r = m["subject_id"].map(u)
    tr, va, _ = cfg.split
    m["split"] = np.where(r < tr, "train", np.where(r < tr + va, "val", "test"))
    keep = ["subject_id", "study_id", "path", "ecg_time", "k", "k_dt_min", "itemid", "age", "gender", "split"]
    m = m[keep].reset_index(drop=True)
    if cfg.max_ecgs:
        m = m.sample(min(cfg.max_ecgs, len(m)), random_state=cfg.seed).reset_index(drop=True)
    return m


def build_labels(cfg, store) -> pd.DataFrame:
    key = f"cache/labels_w{cfg.window_min}_n{cfg.max_ecgs}.parquet"

    def build():
        rl = record_list(cfg, store)
        if "ecg_time" not in rl:
            mm = machine_measurements(cfg, store)[["study_id", "ecg_time"]]
            rl = rl.merge(mm, on="study_id", how="left")
        k, pat = potassium(cfg, store), patients(cfg, store)
        lab = match(rl, k, pat, cfg)
        log.info("Labeled ECGs: %d (%d patients) | K>=%.1f: %.2f%% | K<%.1f: %.2f%%", len(lab),
                 lab.subject_id.nunique(), cfg.hyper_thr, 100 * (lab.k >= cfg.hyper_thr).mean(),
                 cfg.hypo_thr, 100 * (lab.k < cfg.hypo_thr).mean())
        return lab
    return _cached(cfg, store, key, build)


In [ ]:
%%writefile /kaggle/working/repo/src/model.py
"""1D ResNet for 12-lead ECG (12 x 2500 @ 250 Hz) with 3 heads: K+ value, hyperkalemia, hypokalemia."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    def __init__(self, cin, cout, stride, k):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False)
        self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False)
        self.b2 = nn.BatchNorm1d(cout)
        self.down = None if stride == 1 and cin == cout else nn.Sequential(
            nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return F.relu(y + (x if self.down is None else self.down(x)))


class ResNet1D(nn.Module):
    def __init__(self, width=64, kernel=7, dropout=0.3, n_leads=12):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv1d(n_leads, w, 15, 2, 7, bias=False), nn.BatchNorm1d(w), nn.ReLU(),
                                  nn.MaxPool1d(3, 2, 1))
        chans = [w, 2 * w, 4 * w, 8 * w]
        layers, cin = [], w
        for i, c in enumerate(chans):
            layers += [Block(cin, c, 1 if i == 0 else 2, kernel), Block(c, c, 1, kernel)]
            cin = c
        self.features = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(2 * cin, 3)          # [k_z, logit_hyper, logit_hypo]

    def forward(self, x, return_maps=False):
        a = self.features(self.stem(x))
        z = torch.cat([a.mean(-1), a.amax(-1)], 1)
        out = self.head(self.drop(z))
        return (out, a) if return_maps else out


@torch.enable_grad()
def gradcam(model, x, target=1):
    """Grad-CAM over the last residual stage -> (B, T) saliency upsampled to input length, each row sums to 1."""
    model.eval()
    x = x.clone().requires_grad_(False)
    out, a = model(x, return_maps=True)
    a.retain_grad()
    out[:, target].sum().backward()
    w = a.grad.mean(-1, keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-1], mode="linear", align_corners=False)[:, 0]
    tot = cam.sum(-1, keepdim=True)
    cam = torch.where(tot > 0, cam / tot.clamp_min(1e-12), torch.full_like(cam, 1.0 / cam.shape[-1]))  # all-zero -> uniform
    model.zero_grad(set_to_none=True)
    return cam.detach()


In [ ]:
%%writefile /kaggle/working/repo/src/pipeline.py
"""End-to-end: labels -> waveform shards -> resumable training -> test evaluation -> .pth + report."""
from __future__ import annotations

import json
import logging
import shutil
import time
from pathlib import Path

import numpy as np
import torch

from . import evaluate, labels as lab_mod, waveforms
from .config import Config
from .data import ShardSet
from .store import make_store
from .train import Paused, history_table, label_stats, predict, train

log = logging.getLogger("ecgk")


def run(cfg: Config, fetcher=None, labels=None, mm=None) -> dict:
    t0 = time.time()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
    out, priv = Path(cfg.work_dir), Path(cfg.private_dir)
    for d in (out / "model", out / "tables", out / "figures", priv):
        d.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(cfg.seed)
    store = make_store(cfg)

    # %% Labels: K⁺ near each ECG | BigQuery, cached in the bucket; split by patient.
    # show: lab.split.value_counts()
    lab = labels if labels is not None else lab_mod.build_labels(cfg, store)
    log.info("Labels: %d ECGs / %d patients | split %s", len(lab), lab.subject_id.nunique(),
             lab.split.value_counts().to_dict())
    # %% Waveforms → shards (resumable) | Range requests on the PhysioNet zip; every finished pack is committed to the bucket.
    if fetcher is None and cfg.wave_source in ("zip", "auto"):
        from . import zipsource
        try:
            manifest = zipsource.build_packs(cfg, store, lab)
        except Exception as e:
            if cfg.wave_source == "zip":
                raise
            log.warning("ZIP route failed (%s) -> per-file downloads", e)
            manifest = waveforms.build_shards(cfg, store, lab)
    else:
        manifest = waveforms.build_shards(cfg, store, lab, fetcher)
    if cfg.train_needs_gpu and not torch.cuda.is_available():
        log.warning("All waveform shards are ready in the bucket, but this session has no GPU. "
                    "Switch the accelerator to GPU T4 x2 and Run All again to train (shards are reused).")
        return {"status": "shards_ready_no_gpu", "n_shards": sum(len(v) for v in manifest.values())}
    sets = {s: ShardSet(cfg, store, manifest[s]) for s in ("train", "val", "test")}
    stats = label_stats(lab, cfg)
    # %% Train ResNet-1D (resumable) | Full training state is saved every 10 minutes. The table shows every epoch, also the ones loaded from the checkpoint (★ = best).
    # show: history_table(st)
    try:
        model, st = train(cfg, store, sets, stats)
    except Paused as e:
        log.warning("PAUSED: %s -> Run All again (same run_name) to continue.", e)
        return {"status": "paused", "detail": str(e)}
    # %% Evaluate on test | Bootstrap CIs, subgroups, machine-measurement baseline, Grad-CAM beats.
    # show: cls
    # show: reg
    pred = predict(model, sets["test"], cfg, stats)
    cls = evaluate.classification(pred, cfg)
    reg = evaluate.regression(pred)
    sub = evaluate.subgroups(pred, cfg)
    try:
        mmdf = mm if mm is not None else lab_mod.machine_measurements(cfg, store)
        base = evaluate.baseline_machine(lab, mmdf, cfg)
    except Exception as e:
        log.warning("machine-measurement baseline skipped: %s", e)
        base = {}
    beats = evaluate.gradcam_beats(model, sets["test"], pred, cfg)
    evaluate.plots(pred, cls, beats, st["history"], base, cfg, out / "figures")

    # %% Save model + report | `.pth` bundle with everything inference needs.
    bundle = {"state_dict": model.state_dict(), "arch": "ResNet1D",
              "arch_kwargs": dict(width=cfg.width, kernel=cfg.kernel, dropout=cfg.dropout),
              "stats": stats, "fs": cfg.fs_out, "seconds": 10, "leads": waveforms.LEADS,
              "thresholds": dict(hyper=cfg.hyper_thr, severe=cfg.severe_thr, hypo=cfg.hypo_thr)}
    torch.save(bundle, out / "model" / "ecg_k_resnet1d.pth")
    cohort = dict(n_ecgs=int(len(lab)), n_patients=int(lab.subject_id.nunique()),
                  by_split={s: int(n) for s, n in lab.split.value_counts().items()},
                  n_waveforms_used={s: int(sets[s].n) for s in sets},
                  hyper_prevalence=float((lab.k >= cfg.hyper_thr).mean()),
                  hypo_prevalence=float((lab.k < cfg.hypo_thr).mean()),
                  k_median=float(lab.k.median()), abs_dt_median_min=float(lab.k_dt_min.abs().median()))
    metrics = dict(cohort=cohort, classification=cls.to_dict("records"), regression=reg,
                   baseline_machine_measurements=base, training=dict(best_epoch=st["best_epoch"] + 1,
                   best_val_auroc=st["best_auc"], epochs_run=len(st["history"])),
                   gradcam={g: {k: v for k, v in b.items() if k.startswith(("cam_share", "n_beats"))}
                            for g, b in beats.items() if g != "t_ms"},
                   config=cfg.to_dict(), runtime_min=round((time.time() - t0) / 60, 2))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=1, default=float))
    (out / "model" / "model_card.json").write_text(json.dumps(
        {k: bundle[k] for k in ("arch", "arch_kwargs", "stats", "fs", "seconds", "leads", "thresholds")}
        | {"test": metrics["classification"], "regression": reg}, indent=1, default=float))
    cls.to_csv(out / "tables" / "classification.csv", index=False)
    sub.to_csv(out / "tables" / "subgroups.csv", index=False)
    (out / "tables" / "training_history.json").write_text(json.dumps(st["history"], indent=1))
    from .report import build as build_report
    build_report(out)

    # patient-level predictions -> private only
    (priv / "preds").mkdir(exist_ok=True)
    pred[["study_id", "k", "pred_k", "p_hyper", "p_hypo"]].to_parquet(priv / "preds" / "test_predictions.parquet")
    store.put_dir(priv / "preds", f"runs/{cfg.run_name}/private")
    store.put_dir(out, f"runs/{cfg.run_name}/results")
    if cfg.cleanup_private:
        shutil.rmtree(priv, ignore_errors=True)
    a = cls.iloc[0]
    log.info("DONE | test AUROC(%s)=%.3f (%.3f-%.3f) | MAE=%.3f | model -> %s", a.Target, a.AUROC, a["AUROC lo"],
             a["AUROC hi"], reg["mae"], store.uri(f"runs/{cfg.run_name}/results/model/ecg_k_resnet1d.pth"))
    return {"status": "complete", **metrics}


In [ ]:
%%writefile /kaggle/working/repo/src/predict.py
"""Inference on any 12-lead WFDB ECG with a trained .pth bundle.

    python -m src.predict --weights ecg_k_resnet1d.pth --record path/to/40689238      # .hea/.dat pair
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch

from .model import ResNet1D
from .waveforms import clean_resample, parse_wfdb


def load(weights: str, device="cpu"):
    b = torch.load(weights, map_location=device, weights_only=False)
    m = ResNet1D(**b["arch_kwargs"])
    m.load_state_dict(b["state_dict"])
    return m.eval().to(device), b


@torch.no_grad()
def predict_record(model, bundle, hea: bytes, dat: bytes) -> dict:
    x, fs = parse_wfdb(hea, dat)
    x = clean_resample(x, fs, bundle["fs"], bundle["fs"] * bundle["seconds"])
    if x is None:
        raise ValueError("ECG unusable (mostly missing or flat leads)")
    o = model(torch.from_numpy(x.astype(np.float32))[None])[0].numpy()
    s, t = bundle["stats"], bundle["thresholds"]
    return {"predicted_k_mmol_l": round(float(o[0] * s["k_std"] + s["k_mean"]), 2),
            f"p_hyperkalemia_k_ge_{t['hyper']}": round(float(1 / (1 + np.exp(-o[1]))), 4),
            f"p_hypokalemia_k_lt_{t['hypo']}": round(float(1 / (1 + np.exp(-o[2]))), 4)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--record", required=True, help="WFDB record path without extension")
    a = ap.parse_args()
    model, bundle = load(a.weights)
    r = Path(a.record)
    print(json.dumps(predict_record(model, bundle, r.with_suffix(".hea").read_bytes(), r.with_suffix(".dat").read_bytes()), indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/repo/src/report.py
"""GitHub-ready REPORT.md (aggregate only)."""
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


def _t(df: pd.DataFrame) -> str:
    f = lambda v: "–" if isinstance(v, float) and np.isnan(v) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    return "\n".join(["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)] +
                     ["| " + " | ".join(f(v) for v in r) + " |" for r in df.itertuples(index=False)])


def build(work_dir) -> str:
    out = Path(work_dir)
    rep = out / "report"
    (rep / "figures").mkdir(parents=True, exist_ok=True)
    for f in (out / "figures").glob("*.png"):
        shutil.copy(f, rep / "figures" / f.name)
    m = json.loads((out / "metrics.json").read_text())
    c, cfg, reg, base = m["cohort"], m["config"], m["regression"], m["baseline_machine_measurements"]
    cls = pd.DataFrame(m["classification"])
    small = cls["n positive"] < cfg["min_cell"]
    cls["AUROC (95% CI)"] = ["–" if sm or np.isnan(r.AUROC) else f"{r.AUROC:.3f} ({r['AUROC lo']:.3f}–{r['AUROC hi']:.3f})"
                             for sm, (_, r) in zip(small, cls.iterrows())]
    for col in ("prevalence", "AUPRC", "Sens @ 90% spec"):      # prevalence x n would reveal a small count
        cls[col] = cls[col].astype(object).where(~small, np.nan)
    cls["n positive"] = [f"<{cfg['min_cell']}" if sm else v for sm, v in zip(small, cls["n positive"])]
    cls = cls[["Target", "n positive", "prevalence", "AUROC (95% CI)", "AUPRC", "Sens @ 90% spec"]]
    sub = pd.read_csv(out / "tables" / "subgroups.csv")
    hyper = m["classification"][0]
    comp = pd.DataFrame([
        {"Model": "1D ResNet on raw 12-lead ECG (this work)", "AUROC (K ≥ %.1f)" % cfg["hyper_thr"]:
         f"{hyper['AUROC']:.3f} ({hyper['AUROC lo']:.3f}–{hyper['AUROC hi']:.3f})"},
        {"Model": "XGBoost on ECG-machine intervals/axes (baseline)", "AUROC (K ≥ %.1f)" % cfg["hyper_thr"]:
         f"{base['auroc']:.3f} ({base['auroc_lo']:.3f}–{base['auroc_hi']:.3f})" if base else "–"}])
    gc = m.get("gradcam", {})
    gct = pd.DataFrame([{"Group": g, "beats": v["n_beats"], "Grad-CAM share in ST–T (120–450 ms)": v["cam_share_st_t"],
                         "share in QRS (−50–80 ms)": v["cam_share_qrs"]} for g, v in gc.items()])
    md = f"""# Results — Potassium (Hyperkalemia) from the 12-lead ECG

*MIMIC-IV-ECG waveforms + MIMIC-IV potassium within ±{cfg['window_min']} min (hemolyzed samples excluded). Patient-level split.
Aggregate results only; cells with fewer than {cfg['min_cell']} are suppressed.*

## Cohort
| | |
|---|---|
| ECGs with a matched K+ | {c['n_ecgs']:,} |
| Patients | {c['n_patients']:,} |
| Waveforms used (train / val / test) | {c['n_waveforms_used']['train']:,} / {c['n_waveforms_used']['val']:,} / {c['n_waveforms_used']['test']:,} |
| K ≥ {cfg['hyper_thr']} / K < {cfg['hypo_thr']} prevalence | {c['hyper_prevalence']:.2%} / {c['hypo_prevalence']:.2%} |
| Median K+ · median |ECG–lab time gap| | {c['k_median']:.1f} mmol/L · {c['abs_dt_median_min']:.0f} min |

## Test performance (ECG only — no labs, no demographics)
{_t(cls)}

**K+ regression:** MAE {reg['mae']:.3f} mmol/L · RMSE {reg['rmse']:.3f} · Pearson r {reg['pearson_r']:.3f} · within ±0.5 mmol/L {reg['within_0_5']:.1%}

![ROC](figures/roc.png)
![Predicted vs measured](figures/pred_vs_true.png)

## Deep model vs classical ECG measurements
{_t(comp)}

## Where does the model look? (population Grad-CAM, R-peak aligned)
{_t(gct) if len(gct) else '–'}

![Grad-CAM beat](figures/gradcam_beat.png)

## Subgroups (K ≥ {cfg['hyper_thr']})
{_t(sub)}

## Training
Best epoch {m['training']['best_epoch']} of {m['training']['epochs_run']} · best val AUROC {m['training']['best_val_auroc']:.3f}

![Training](figures/training.png)
"""
    (rep / "REPORT.md").write_text(md, encoding="utf-8")
    return md


In [ ]:
%%writefile /kaggle/working/repo/src/store.py
"""Checkpoint store: a private GCS bucket (auto-created) or a local folder (tests / no GCP).
Everything needed to resume after a Kaggle session stops lives here."""
from __future__ import annotations

import logging
import re
import shutil
from pathlib import Path

log = logging.getLogger(__name__)


def default_bucket_name(project_id: str) -> str:
    name = re.sub(r"[^a-z0-9-]", "-", f"{project_id}-mimic-ckpt".lower()).strip("-")
    return name[:63]


class GCSStore:
    def __init__(self, project_id: str, bucket: str, prefix: str, location: str = "US"):
        from google.cloud import storage

        from .gcp import credentials
        self.client = storage.Client(project=project_id, credentials=credentials("gcs"))
        self.name = (bucket or default_bucket_name(project_id)).replace("gs://", "").strip("/")
        self.prefix = prefix.strip("/")
        self.bucket = self._ensure(project_id, location)

    def _ensure(self, project_id, location):
        from google.api_core import exceptions as gexc
        b = self.client.bucket(self.name)
        try:
            if b.exists():
                return b
        except gexc.Forbidden:
            return b           # exists but owned elsewhere / no get permission; writes will tell
        try:
            b.iam_configuration.uniform_bucket_level_access_enabled = True
            b.iam_configuration.public_access_prevention = "enforced"   # never public
            b = self.client.create_bucket(b, project=project_id, location=location)
            log.info("Created private bucket gs://%s (%s)", self.name, location)
        except gexc.Conflict:
            raise RuntimeError(f"Bucket name '{self.name}' is taken globally. Set cfg.bucket to another name.")
        except Exception as e:
            raise RuntimeError(
                f"Could not create bucket gs://{self.name} ({e}). Create it once in Cloud Shell:\n"
                f"  gcloud storage buckets create gs://{self.name} --project={project_id} --location={location} "
                f"--uniform-bucket-level-access --public-access-prevention\nthen set cfg.bucket='{self.name}'.")
        return b

    def uri(self, key=""):
        return f"gs://{self.name}/{self.prefix}/{key}".rstrip("/")

    def _k(self, key):
        return f"{self.prefix}/{key}"

    def put(self, local, key):
        self.bucket.blob(self._k(key)).upload_from_filename(str(local))

    def get(self, key, local) -> bool:
        bl = self.bucket.blob(self._k(key))
        if not bl.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        bl.download_to_filename(str(local))
        return True

    def exists(self, key) -> bool:
        return self.bucket.blob(self._k(key)).exists()

    def list(self, sub) -> list[str]:
        p = self._k(sub).rstrip("/") + "/"
        return [b.name[len(self.prefix) + 1:] for b in self.client.list_blobs(self.bucket, prefix=p)]

    def put_dir(self, local_dir, sub):
        n = 0
        for p in sorted(Path(local_dir).rglob("*")):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")
                n += 1
        log.info("Uploaded %d files -> %s", n, self.uri(sub))


class LocalStore:
    """Same interface backed by a folder (unit tests, or running without GCP)."""

    def __init__(self, root, prefix=""):
        self.root = Path(root) / prefix
        self.root.mkdir(parents=True, exist_ok=True)
        self.name = str(self.root)

    def uri(self, key=""):
        return str(self.root / key)

    def put(self, local, key):
        dst = self.root / key
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local, dst)

    def get(self, key, local) -> bool:
        src = self.root / key
        if not src.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        if Path(local).resolve() != src.resolve():
            shutil.copyfile(src, local)
        return True

    def exists(self, key) -> bool:
        return (self.root / key).exists()

    def list(self, sub) -> list[str]:
        d = self.root / sub
        return [p.relative_to(self.root).as_posix() for p in d.rglob("*") if p.is_file()] if d.exists() else []

    def put_dir(self, local_dir, sub):
        for p in Path(local_dir).rglob("*"):
            if p.is_file():
                self.put(p, f"{sub.strip('/')}/{p.relative_to(local_dir).as_posix()}")


def make_store(cfg):
    if getattr(cfg, "local_store", ""):
        return LocalStore(cfg.local_store, cfg.gcs_prefix)
    from .gcp import validate_project_id
    cfg.project_id = validate_project_id(cfg.project_id)
    s = GCSStore(cfg.project_id, cfg.bucket, cfg.gcs_prefix, cfg.bucket_location)
    cfg.bucket = s.name
    log.info("Checkpoints -> %s", s.uri())
    return s


In [ ]:
%%writefile /kaggle/working/repo/src/train.py
"""Resumable training: full state (model/optimizer/scheduler/AMP scaler/epoch/shard position/early stopping)
is checkpointed to the store every `ckpt_minutes` and at every epoch end. Re-run = continue exactly there."""
from __future__ import annotations

import json
import logging
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

from .model import ResNet1D

log = logging.getLogger(__name__)


class Paused(Exception):
    """Session time budget used up; state is checkpointed, re-run to continue."""


def device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_model(cfg):
    return ResNet1D(cfg.width, cfg.kernel, cfg.dropout)


def label_stats(labels: pd.DataFrame, cfg) -> dict:
    tr = labels[labels.split == "train"]
    hyper, hypo = (tr.k >= cfg.hyper_thr).mean(), (tr.k < cfg.hypo_thr).mean()
    return dict(k_mean=float(tr.k.mean()), k_std=float(tr.k.std() or 1.0),
                pw_hyper=float(np.clip((1 - hyper) / max(hyper, 1e-6), 1, 30)),
                pw_hypo=float(np.clip((1 - hypo) / max(hypo, 1e-6), 1, 30)))


def _augment(x):
    b = x.shape[0]
    x = x * torch.empty(b, 12, 1, device=x.device).uniform_(0.85, 1.15)
    x = x + torch.randn_like(x) * torch.empty(b, 1, 1, device=x.device).uniform_(0.0, 0.03)
    keep = (torch.rand(b, 12, 1, device=x.device) > 0.1).float()
    x = x * keep
    shift = int(torch.randint(-250, 251, (1,)))
    return torch.roll(x, shift, dims=-1)


def _loss(out, k, stats, cfg):
    kz = (k - stats["k_mean"]) / stats["k_std"]
    hyper, hypo = (k >= cfg.hyper_thr).float(), (k < cfg.hypo_thr).float()
    dev = out.device
    return (F.huber_loss(out[:, 0].float(), kz, delta=1.0)
            + F.binary_cross_entropy_with_logits(out[:, 1].float(), hyper, pos_weight=torch.tensor(stats["pw_hyper"], device=dev))
            + 0.5 * F.binary_cross_entropy_with_logits(out[:, 2].float(), hypo, pos_weight=torch.tensor(stats["pw_hypo"], device=dev)))


@torch.no_grad()
def predict(model, shardset, cfg, stats, bs=512) -> pd.DataFrame:
    model.eval()
    dev = device()
    rows = []
    for _, _, X, meta in shardset.iterate(range(len(shardset.entries))):
        outs = []
        for b in range(0, len(X), bs):
            xb = torch.from_numpy(np.asarray(X[b:b + bs], dtype=np.float32)).to(dev)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=cfg.amp and dev.type == "cuda"):
                outs.append(model(xb).float().cpu())
        o = torch.cat(outs).numpy() if outs else np.zeros((0, 3))
        m = meta.copy()
        m["pred_k"] = o[:, 0] * stats["k_std"] + stats["k_mean"]
        m["p_hyper"] = 1 / (1 + np.exp(-o[:, 1]))
        m["p_hypo"] = 1 / (1 + np.exp(-o[:, 2]))
        rows.append(m)
    return pd.concat(rows, ignore_index=True)


def train(cfg, store, sets: dict, stats: dict):
    dev = device()
    ck = f"ckpt/{cfg.run_name}"
    loc = Path(cfg.private_dir) / "ckpt"
    loc.mkdir(parents=True, exist_ok=True)
    model = make_model(cfg).to(dev)
    net = torch.nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    tr = sets["train"]
    spe = sum(math.ceil(n / cfg.batch_size) for _, n in tr.entries)
    total, warm = cfg.epochs * spe, max(1, int(cfg.warmup_frac * cfg.epochs * spe))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and dev.type == "cuda")
    st = dict(epoch=0, shards_done=0, step=0, best_auc=-1.0, best_epoch=-1, bad_epochs=0, history=[],
              finished=False, stats=stats)

    if store.get(f"{ck}/last.pt", loc / "last.pt"):
        s = torch.load(loc / "last.pt", map_location=dev, weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        scaler.load_state_dict(s["scaler"]); st = s["state"]
        log.info("loaded checkpoint: %d of %d epochs already done%s", len(st["history"]), cfg.epochs,
                 " (training finished)" if st["finished"] else "")
        log.info("RESUME training: epoch %d, shard %d/%d, step %d, best val AUROC %.4f (epoch %d)",
                 st["epoch"] + 1, st["shards_done"], len(tr.entries), st["step"], st["best_auc"], st["best_epoch"] + 1)

    loaded_epochs = len(st["history"])

    def save(tag="last"):
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "state": st}, loc / "last.pt")
        store.put(loc / "last.pt", f"{ck}/last.pt")
        log.info("checkpoint (%s): epoch %d shard %d step %d", tag, st["epoch"] + 1, st["shards_done"], st["step"])

    def save_best():
        torch.save(model.state_dict(), loc / "best.pth")
        store.put(loc / "best.pth", f"{ck}/best.pth")

    if st["finished"]:
        store.get(f"{ck}/best.pth", loc / "best.pth")
        model.load_state_dict(torch.load(loc / "best.pth", map_location=dev))
        return model, {**st, "loaded_epochs": loaded_epochs}

    t0 = last_ck = time.time()
    deadline = t0 + cfg.max_runtime_hours * 3600
    shards_session = 0
    log.info("Training on %s x%d | %d train ECGs, %d steps/epoch", dev, max(torch.cuda.device_count(), 1), tr.n, spe)
    while st["epoch"] < cfg.epochs:
        ep = st["epoch"]
        order = np.random.default_rng(cfg.seed + ep).permutation(len(tr.entries))
        net.train()
        losses = []
        for i, key, X, meta in tr.iterate(order[st["shards_done"]:]):
            perm = np.random.default_rng(cfg.seed + 100003 * ep + int(i)).permutation(len(X))
            kk = meta["k"].to_numpy(np.float32)
            for b in range(0, len(perm), cfg.batch_size):
                idx = np.sort(perm[b:b + cfg.batch_size])
                xb = _augment(torch.from_numpy(X[idx]).to(dev, non_blocking=True).float())   # fp16 over PCIe
                kb = torch.from_numpy(kk[idx]).to(dev)
                with torch.autocast(dev.type, dtype=torch.float16, enabled=cfg.amp and dev.type == "cuda"):
                    out = net(xb)
                loss = _loss(out, kb, stats, cfg)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt); scaler.update(); sched.step()
                st["step"] += 1
                losses.append(float(loss.detach()))
            st["shards_done"] += 1
            shards_session += 1
            if time.time() - last_ck >= cfg.ckpt_minutes * 60:
                save("periodic"); last_ck = time.time()
            if cfg.debug_crash_after_shards is not None and shards_session >= cfg.debug_crash_after_shards:
                raise RuntimeError("simulated session death")
            if time.time() > deadline:
                save("time budget")
                raise Paused(f"paused at epoch {ep + 1}, shard {st['shards_done']}")
        # ---- epoch end: validate ----
        pv = predict(model, sets["val"], cfg, stats)
        y = (pv.k >= cfg.hyper_thr).astype(int)
        auc = float(roc_auc_score(y, pv.p_hyper)) if 0 < y.sum() < len(y) else float("nan")
        mae = float(np.mean(np.abs(pv.pred_k - pv.k)))
        st["history"].append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)) if losses else None,
                                  val_auroc_hyper=auc, val_mae=mae, lr=float(sched.get_last_lr()[0])))
        log.info("epoch %d/%d | train loss %.4f | val AUROC(K>=%.1f) %.4f | val MAE %.3f",
                 ep + 1, cfg.epochs, st["history"][-1]["train_loss"] or float("nan"), cfg.hyper_thr, auc, mae)
        if auc > st["best_auc"]:
            st.update(best_auc=auc, best_epoch=ep, bad_epochs=0)
            save_best()
        else:
            st["bad_epochs"] += 1
        st["epoch"], st["shards_done"] = ep + 1, 0
        stop = st["bad_epochs"] >= cfg.patience
        if stop or st["epoch"] >= cfg.epochs:
            st["finished"] = True
        save("epoch end"); last_ck = time.time()
        if stop:
            log.info("early stopping (no improvement for %d epochs)", cfg.patience)
            break
    st["finished"] = True
    save("finished")
    model.load_state_dict(torch.load(loc / "best.pth", map_location=dev))
    (loc / "history.json").write_text(json.dumps(st["history"], indent=1))
    return model, {**st, "loaded_epochs": loaded_epochs}


def history_table(st) -> pd.DataFrame:
    """Every epoch (also those restored from the checkpoint); ★ = best val AUROC, used for the final model."""
    h = pd.DataFrame(st.get("history", []))
    if not len(h):
        return pd.DataFrame({"info": ["no epochs finished yet"]})
    h["source"] = np.where(h.epoch <= st.get("loaded_epochs", 0), "checkpoint", "this session")
    h["best"] = np.where(h.epoch == st.get("best_epoch", -1) + 1, "★", "")
    return h


In [ ]:
%%writefile /kaggle/working/repo/src/waveforms.py
"""Download MIMIC-IV-ECG WFDB records in parallel, parse, resample 500->250 Hz, write float16 shards.
Shards go to the checkpoint store; finished shards are skipped on re-run (resumable)."""
from __future__ import annotations

import io
import json
import logging
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)
LEADS = ["I", "II", "III", "AVR", "AVL", "AVF", "V1", "V2", "V3", "V4", "V5", "V6"]
_GAIN = re.compile(r"^([-\d.eE+]+)(?:\(([-\d]+)\))?(?:/(\w+))?$")


def parse_wfdb(hea: bytes, dat: bytes) -> tuple[np.ndarray, float]:
    """Return (12, n) float32 signal in mV in standard lead order, and fs. Supports format 16 natively."""
    lines = [l for l in hea.decode("latin-1").splitlines() if l.strip() and not l.startswith("#")]
    head = lines[0].split()
    nsig, fs = int(head[1]), float(head[2].split("/")[0])
    nsamp = int(head[3]) if len(head) > 3 else None
    sig = [l.split() for l in lines[1:1 + nsig]]
    fmts = {s[1].split("x")[0].split(":")[0] for s in sig}
    if fmts != {"16"}:
        return _parse_with_wfdb(hea, dat)
    raw = np.frombuffer(dat, dtype="<i2")
    if nsamp is None:
        nsamp = len(raw) // nsig
    raw = raw[: nsamp * nsig].reshape(nsamp, nsig).astype(np.float32)
    out = np.zeros((12, nsamp), np.float32)
    for j, s in enumerate(sig):
        m = _GAIN.match(s[2])
        gain = float(m.group(1)) if m else 200.0
        gain = gain if gain != 0 else 200.0
        adczero = int(s[4]) if len(s) > 4 and s[4].lstrip("-").isdigit() else 0
        base = int(m.group(2)) if m and m.group(2) is not None else adczero
        unit = (m.group(3) or "mV") if m else "mV"
        x = raw[:, j]
        x[x == -32768] = np.nan
        x = (x - base) / gain
        if unit.lower() in ("uv", "µv"):
            x = x / 1000.0
        name = s[-1].upper() if len(s) >= 9 else LEADS[j] if j < 12 else ""
        if name in LEADS:
            out[LEADS.index(name)] = x
    return out, fs


def _parse_with_wfdb(hea: bytes, dat: bytes):
    import tempfile

    import wfdb
    with tempfile.TemporaryDirectory() as d:
        rec = hea.decode("latin-1").split()[0]
        (Path(d) / f"{rec}.hea").write_bytes(hea)
        (Path(d) / f"{rec}.dat").write_bytes(dat)
        sig, f = wfdb.rdsamp(str(Path(d) / rec))
    names = [n.upper() for n in f["sig_name"]]
    out = np.zeros((12, sig.shape[0]), np.float32)
    for j, n in enumerate(names):
        if n in LEADS:
            out[LEADS.index(n)] = sig[:, j]
    return out, float(f["fs"])


def clean_resample(x: np.ndarray, fs: float, fs_out: int, n_out: int) -> np.ndarray | None:
    """(12, n) mV -> (12, n_out) float16, median-centred, NaN->0, clipped. None if unusable."""
    from scipy.signal import resample_poly
    if np.isnan(x).mean() > 0.5:
        return None
    x = np.nan_to_num(x, nan=0.0)
    if int(fs) != fs_out:
        from math import gcd
        g = gcd(int(fs), fs_out)
        x = resample_poly(x, fs_out // g, int(fs) // g, axis=1)
    x = x[:, :n_out]
    if x.shape[1] < n_out:
        x = np.pad(x, ((0, 0), (0, n_out - x.shape[1])))
    x = x - np.median(x, axis=1, keepdims=True)
    if (np.abs(x).max(axis=1) < 1e-3).sum() > 6:          # >6 flat leads
        return None
    return np.clip(x, -10, 10).astype(np.float16)


def s3_list(bucket_url: str, prefix: str, timeout=30, max_pages=20) -> tuple[list[str], list[str]]:
    """Anonymous S3 ListObjectsV2 (paginated) -> (sub-prefixes, keys) directly under `prefix`."""
    import xml.etree.ElementTree as ET

    import requests
    ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
    pre, keys, token = [], [], None
    for _ in range(max_pages):
        params = {"list-type": "2", "prefix": prefix, "delimiter": "/", "max-keys": "1000"}
        if token:
            params["continuation-token"] = token
        r = requests.get(bucket_url, params=params, timeout=timeout)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        pre += [e.text for e in root.findall("s:CommonPrefixes/s:Prefix", ns)]
        keys += [e.text for e in root.findall("s:Contents/s:Key", ns)]
        t = root.find("s:NextContinuationToken", ns)
        if t is None or not t.text:
            break
        token = t.text
    return pre, keys


def discover_s3_base(cfg, record_path: str) -> str | None:
    """Find the S3 prefix under which `record_path` (files/pNNNN/...) exists. Returns a base URL or None.
    Searches the bucket's top level for folders whose name contains the dataset name (or 'ecg')."""
    import requests
    seen = []
    try:
        top, _ = s3_list(cfg.s3_bucket_url, "")
    except Exception as e:
        log.warning("S3 top-level listing failed: %s", e)
        top = []
    name = cfg.s3_dataset.lower()
    starts = [p for p in top if name in p.lower()] + [p for p in top if "ecg" in p.lower() and name not in p.lower()]
    seen.append(f"top-level matches: {starts[:15]} (of {len(top)} folders)")
    frontier = starts[:10] or [name + "/"]
    for _ in range(3):
        nxt = []
        for p in frontier:
            try:
                if requests.head(cfg.s3_bucket_url + p + record_path + ".hea", timeout=15).status_code == 200:
                    log.info("S3 prefix found: %s", p)
                    return cfg.s3_bucket_url + p
            except Exception:
                pass
            try:
                sub, _ = s3_list(cfg.s3_bucket_url, p)
                seen.append(f"{p} -> {sub[:8]}")
                nxt += [x for x in sub if not x.endswith("files/")][:20]
            except Exception as e:
                seen.append(f"{p} -> list failed: {type(e).__name__}")
        frontier = nxt
        if not frontier:
            break
    log.warning("S3 prefix for %s not found. %s", record_path, seen)
    return None


class HttpFetcher:
    """Thread-safe downloader. Probes the mirrors once, uses the fastest working one; physionet.org (throttled)
    is only a fallback with limited concurrency. Errors are counted so failures are visible, never silent."""

    def __init__(self, bases, workers, org_workers=6, probe_path=None):
        import collections
        self.bases = list(bases)
        self.local = threading.local()
        self.workers = workers
        self.org_sem = threading.Semaphore(org_workers)
        self.errors = collections.Counter()
        self.lock = threading.Lock()
        if probe_path:
            self.bases = self.probe(probe_path)

    def _session(self):
        if not hasattr(self.local, "s"):
            import requests
            from requests.adapters import HTTPAdapter
            from urllib3.util.retry import Retry
            s = requests.Session()
            s.mount("https://", HTTPAdapter(pool_connections=8, pool_maxsize=self.workers, max_retries=Retry(
                total=8, connect=6, read=3, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504))))
            self.local.s = s
        return self.local.s

    def probe(self, path):
        ok, report = [], []
        for b in self.bases:
            t = time.time()
            try:
                r = self._session().get(b + path + ".hea", timeout=(10, 30))
                report.append(f"{b} -> HTTP {r.status_code} in {time.time() - t:.2f}s")
                if r.ok:
                    ok.append((time.time() - t, b))
            except Exception as e:
                report.append(f"{b} -> {type(e).__name__}: {str(e)[:120]}")
        for line in report:
            log.info("mirror probe: %s", line)
        if not ok:
            raise RuntimeError("No ECG mirror reachable from this machine:\n  " + "\n  ".join(report) +
                               "\nCheck that Internet is ON in the Kaggle notebook settings.")
        fast = [b for _, b in sorted(ok)]
        org = [b for b in fast if "physionet.org" in b]
        chosen = [b for b in fast if "physionet.org" not in b][:1] + org[:1]
        log.info("Using mirror(s): %s", chosen)
        return chosen

    def _get(self, base, url):
        if "physionet.org" in base:
            with self.org_sem:
                return self._session().get(url, timeout=(30, 90))
        return self._session().get(url, timeout=(10, 60))

    def __call__(self, path: str) -> tuple[bytes, bytes]:
        last = None
        for b in self.bases:
            try:
                h = self._get(b, b + path + ".hea")
                h.raise_for_status()
                d = self._get(b, b + path + ".dat")
                d.raise_for_status()
                return h.content, d.content
            except Exception as e:
                last = e
                code = getattr(getattr(e, "response", None), "status_code", None)
                with self.lock:
                    self.errors[f"{b.split('/')[2]}: {code or type(e).__name__}"] += 1
        raise RuntimeError(f"{path}: {last}")


def _eta(labels, split, ci, size, rate):
    order = ["train", "val", "test"]
    left = sum(int((labels.split == s).sum()) for s in order[order.index(split) + 1:])
    left += max(int((labels.split == split).sum()) - (ci + 1) * size, 0)
    m = left / max(rate, 1e-6) / 60
    return f"{m:.0f} min" if m < 120 else f"{m / 60:.1f} h"


def build_shards(cfg, store, labels: pd.DataFrame, fetcher=None) -> dict:
    """Writes shards/<split>/shard_XXXXX.{npy,parquet}; returns manifest {split: [(key, n), ...]}."""
    man_key = f"shards/manifest_n{len(labels)}.json"
    man_loc = Path(cfg.private_dir) / man_key
    if store.get(man_key, man_loc):
        return json.loads(man_loc.read_text())
    if fetcher is None:
        bases = list(cfg.wave_base_urls)
        found = discover_s3_base(cfg, labels.path.iloc[0])
        if found:
            bases = [found] + [b for b in bases if b != found]
        fetcher = HttpFetcher(bases, cfg.download_workers, cfg.physionet_org_workers, probe_path=labels.path.iloc[0])
        if all("physionet.org" in b for b in fetcher.bases):
            log.warning("Only physionet.org is reachable: downloads will be slow (throttled). "
                        "Progress is saved per shard, so re-running continues where it stopped.")
    n_out = 10 * cfg.fs_out
    tmp = Path(cfg.private_dir) / "shard_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    manifest, t0, done_ecgs = {}, time.time(), 0
    existing = set(store.list("shards"))
    for split in ("train", "val", "test"):
        d = labels[labels.split == split].sample(frac=1.0, random_state=cfg.seed).reset_index(drop=True)
        chunks = [d.iloc[i:i + cfg.shard_size] for i in range(0, len(d), cfg.shard_size)]
        manifest[split] = []
        for ci, ch in enumerate(chunks):
            key = f"shards/{split}/shard_{ci:05d}"
            if f"{key}.parquet" in existing and f"{key}.npy" in existing:
                loc = tmp / f"{split}_{ci}.parquet"
                store.get(f"{key}.parquet", loc)
                n_ok = int(len(pd.read_parquet(loc)))
                # shards written by this version carry a .json receipt; older ones must be ~complete
                if f"{key}.json" in existing or n_ok >= 0.95 * len(ch):
                    manifest[split].append([key, n_ok])
                    continue
                log.warning("%s has only %d/%d ECGs (written by an older run) -> rebuilding", key, n_ok, len(ch))

            failed = []

            def work(row):
                try:
                    raw = fetcher(row.path)
                except Exception as e:                 # download problem -> counts against the shard
                    failed.append(str(e)[:200])
                    return None
                try:
                    x, fs = parse_wfdb(*raw)
                    return clean_resample(x, fs, cfg.fs_out, n_out)   # None = unusable signal (expected, rare)
                except Exception:
                    return None
            with ThreadPoolExecutor(cfg.download_workers) as ex:
                res = list(ex.map(work, ch.itertuples(index=False)))
            if len(failed) > cfg.max_fail_frac * len(ch):
                errs = dict(getattr(fetcher, "errors", {}))
                raise RuntimeError(f"{key}: {len(failed)}/{len(ch)} downloads failed -> shard NOT saved "
                                   f"(re-run to retry). Error counts: {errs}. Example: {failed[0]}")
            ok = [i for i, r in enumerate(res) if r is not None]
            X = np.stack([res[i] for i in ok]) if ok else np.zeros((0, 12, n_out), np.float16)
            meta = ch.iloc[ok].drop(columns=["path"]).reset_index(drop=True)
            np.save(tmp / "x.npy", X)
            meta.to_parquet(tmp / "m.parquet", index=False)
            store.put(tmp / "x.npy", f"{key}.npy")
            (tmp / "r.json").write_text(json.dumps({"n": len(ch), "ok": len(ok), "download_failed": len(failed)}))
            store.put(tmp / "r.json", f"{key}.json")
            store.put(tmp / "m.parquet", f"{key}.parquet")      # parquet last = commit marker
            manifest[split].append([key, len(ok)])
            done_ecgs += len(ch)
            el = time.time() - t0
            log.info("%s shard %d/%d: %d/%d ok (%d download errors) | %.0f ECG/s | ETA %s", split, ci + 1, len(chunks),
                     len(ok), len(ch), len(failed), done_ecgs / max(el, 1e-6),
                     _eta(labels, split, ci, cfg.shard_size, done_ecgs / max(el, 1e-6)))
    man_loc.parent.mkdir(parents=True, exist_ok=True)
    man_loc.write_text(json.dumps(manifest))
    store.put(man_loc, man_key)
    return manifest


def synthetic_record(k: float, seed: int = 0, fs=500, n=5000) -> tuple[bytes, bytes]:
    """WFDB bytes for tests/demo: ECG-like signal whose T-wave grows with K+ (no real patient data)."""
    r = np.random.default_rng(seed)
    t = np.arange(n) / fs
    hr = r.uniform(55, 95)
    beat = (t * hr / 60) % 1.0
    qrs = np.exp(-((beat - 0.2) / 0.012) ** 2)
    tw = np.exp(-((beat - 0.45) / (0.05 - 0.004 * (k - 4))) ** 2) * (0.15 + 0.12 * (k - 4))
    sig = np.stack([(qrs * r.uniform(0.6, 1.4) + tw + r.normal(0, 0.02, n)) for _ in range(12)], 1)
    dig = np.round(sig * 200).astype("<i2")
    hdr = [f"rec 12 {fs} {n}"] + [f"rec.dat 16 200/mV 16 0 {dig[0, j]} 0 0 {LEADS[j].replace('AV', 'aV')}" for j in range(12)]
    return ("\n".join(hdr) + "\n").encode(), dig.tobytes()


def check_source(cfg, path="files/p1000/p10000032/s40689238/40689238") -> None:
    """Pre-flight: which open MIMIC-IV-ECG mirrors are reachable from this machine (and how fast)?"""
    import requests
    if cfg.wave_source in ("zip", "auto"):
        from . import zipsource
        try:
            url = zipsource.find_zip_url(cfg)
            size = zipsource.remote_size(url)
            t = time.time()
            b = zipsource.fetch_range(zipsource._session(), url, size - 8 * 2 ** 20, size - 1)
            print(f"  ECG ZIP: {url}\n           {size / 1e9:.1f} GB, Range OK, {len(b) / 1e6 / (time.time() - t):.1f} MB/s on one connection")
        except Exception as e:
            print(f"  ECG ZIP: FAILED -> {type(e).__name__}: {str(e)[:200]}")
        return
    try:
        top, _ = s3_list(cfg.s3_bucket_url, "")
        hits = [p for p in top if "ecg" in p.lower() or "mimic" in p.lower()]
        print(f"  S3 bucket top level: {len(top)} folders; mimic/ecg-like: {hits[:20]}")
    except Exception as e:
        print(f"  S3 listing failed: {type(e).__name__}: {str(e)[:150]}")
    b = discover_s3_base(cfg, path)
    print(f"  S3 auto-discovered base: {b or 'NOT FOUND'}")
    for b in cfg.wave_base_urls:
        t = time.time()
        try:
            r = requests.get(b + path + ".hea", timeout=(10, 30))
            print(f"  ECG mirror {b}: HTTP {r.status_code} ({time.time() - t:.2f}s)")
        except Exception as e:
            print(f"  ECG mirror {b}: FAILED -> {type(e).__name__}: {str(e)[:150]}")


In [ ]:
%%writefile /kaggle/working/repo/src/zipsource.py
"""Fast path for MIMIC-IV-ECG: read the single PhysioNet ZIP with HTTP Range requests.

Instead of ~470k small-file requests (physionet.org throttles those), we
  1. read only the ZIP central directory (index of all members),
  2. group the needed ECGs into ~256 MB contiguous byte ranges,
  3. download each range with a few parallel connections and extract the members from memory.
Every finished range becomes a 'pack' in the checkpoint store -> fully resumable."""
from __future__ import annotations

import io
import json
import logging
import re
import struct
import threading
import time
import zipfile
import zlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

from .waveforms import clean_resample, parse_wfdb

log = logging.getLogger(__name__)
PROJECT_PAGE = "https://physionet.org/content/mimic-iv-ecg/1.0/"


def _session():
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    s = requests.Session()
    s.mount("https://", HTTPAdapter(pool_maxsize=16, max_retries=Retry(
        total=10, connect=8, read=5, backoff_factor=2, status_forcelist=(429, 500, 502, 503, 504))))
    return s


def remote_size(url) -> int:
    r = _session().head(url, timeout=(30, 60), allow_redirects=True)
    r.raise_for_status()
    return int(r.headers["Content-Length"])


ZIP_CANDIDATES = [
    "https://physionet.org/content/mimic-iv-ecg/get-zip/1.0/",
    "https://physionet.org/static/published-projects/mimic-iv-ecg/"
    "mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0.zip",
]


def _resolve(url) -> str | None:
    """Follow redirects; return the final URL if it is a big file that supports Range requests."""
    s = _session()
    try:
        r = s.get(url, headers={"Range": "bytes=0-3"}, timeout=(30, 60), allow_redirects=True, stream=True)
        ok = r.status_code == 206 and r.raw.read(4) == b"PK\x03\x04"
        total = int(r.headers.get("Content-Range", "/0").split("/")[-1] or 0)
        r.close()
        if ok and total > 1e9:
            return r.url
        log.info("zip candidate %s -> HTTP %s, %.1f GB", url, r.status_code, total / 1e9)
    except Exception as e:
        log.info("zip candidate %s -> %s", url, type(e).__name__)
    return None


def find_zip_url(cfg) -> str:
    if cfg.zip_url:
        return _resolve(cfg.zip_url) or cfg.zip_url
    cands = []
    try:
        html = _session().get(PROJECT_PAGE, timeout=(30, 60)).text
        for h in re.findall(r"""href=["']([^"']+)["']""", html):
            if "get-zip" in h or h.lower().endswith(".zip"):
                cands.append(h if h.startswith("http") else "https://physionet.org" + h)
    except Exception as e:
        log.warning("could not read %s: %s", PROJECT_PAGE, e)
    for u in dict.fromkeys(cands + ZIP_CANDIDATES):
        final = _resolve(u)
        if final:
            log.info("ZIP: %s", final)
            return final
    raise RuntimeError(f"No downloadable ZIP found (tried {list(dict.fromkeys(cands + ZIP_CANDIDATES))}). "
                       "Open the PhysioNet page, copy the 'Download the ZIP file' link and set cfg.zip_url.")


class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP Range requests (enough for zipfile to read the index)."""

    def __init__(self, url, session=None):
        self.url, self.s, self.pos = url, session or _session(), 0
        self.size = remote_size(url)

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        if n == 0 or self.pos >= self.size:
            return b""
        end = min(self.pos + n, self.size) - 1
        data = fetch_range(self.s, self.url, self.pos, end)
        self.pos += len(data)
        return data

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


_LIVE = [0]                                    # bytes received by all range fetches (progress heartbeat)


def fetch_range(s, url, start, end, tries=8) -> bytes:
    """Bytes [start, end] inclusive; resumes a broken transfer from where it stopped."""
    buf, pos, attempt = bytearray(), start, 0
    while pos <= end:
        try:
            r = s.get(url, headers={"Range": f"bytes={pos}-{end}"}, stream=True, timeout=(30, 120))
            if r.status_code != 206:
                raise RuntimeError(f"expected HTTP 206, got {r.status_code}")
            for chunk in r.iter_content(1 << 20):
                buf += chunk
                pos += len(chunk)
                _LIVE[0] += len(chunk)
            if pos <= end:
                raise IOError("short read")
        except Exception as e:
            attempt += 1
            if attempt >= tries:
                raise
            log.warning("range %d-%d interrupted at %d (%s), retrying", start, end, pos, type(e).__name__)
            time.sleep(min(60, 2 ** attempt))
    return bytes(buf)


def zip_index(cfg, store) -> pd.DataFrame:
    """Central directory -> DataFrame(path, ext, header_offset, compress_size, compress_type), cached."""
    key = "cache/zip_index.parquet"
    loc = Path(cfg.private_dir) / key
    if loc.exists() or store.get(key, loc):
        return pd.read_parquet(loc)
    url = find_zip_url(cfg)
    log.info("Reading ZIP index of %s ...", url)
    f = HTTPRangeFile(url)
    z = zipfile.ZipFile(io.BufferedReader(f, buffer_size=8 << 20))
    rows = []
    for zi in z.infolist():
        m = re.search(r"(files/p\d+/p\d+/s\d+/\d+)\.(hea|dat)$", zi.filename)
        if m:
            rows.append((m.group(1), m.group(2), zi.header_offset, zi.compress_size, zi.compress_type))
    df = pd.DataFrame(rows, columns=["path", "ext", "header_offset", "compress_size", "compress_type"])
    df.attrs["url"] = url
    log.info("ZIP index: %d members (%.1f GB zip)", len(df), f.size / 1e9)
    loc.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(loc, index=False)
    (loc.parent / "zip_url.txt").write_text(url)
    store.put(loc, key)
    store.put(loc.parent / "zip_url.txt", "cache/zip_url.txt")
    return df


def _member(buf: bytes, base: int, off: int, csize: int, ctype: int) -> bytes:
    p = off - base
    sig, = struct.unpack_from("<I", buf, p)
    if sig != 0x04034B50:
        raise ValueError("bad local header")
    n, m = struct.unpack_from("<HH", buf, p + 26)
    data = buf[p + 30 + n + m: p + 30 + n + m + csize]
    if ctype == zipfile.ZIP_STORED:
        return data
    if ctype == zipfile.ZIP_DEFLATED:
        return zlib.decompressobj(-15).decompress(data)
    raise ValueError(f"compression {ctype} not supported")


def build_packs(cfg, store, labels: pd.DataFrame) -> dict:
    """Writes shards/<split>/pack_XXXXX.{npy,parquet,json}; returns manifest {split: [(key, n)]}."""
    man_key = f"shards/manifest_zip_n{len(labels)}.json"
    man_loc = Path(cfg.private_dir) / man_key
    if store.get(man_key, man_loc):
        return json.loads(man_loc.read_text())
    idx = zip_index(cfg, store)
    url = (Path(cfg.private_dir) / "cache" / "zip_url.txt")
    url = url.read_text().strip() if url.exists() or store.get("cache/zip_url.txt", url) else find_zip_url(cfg)
    piv = idx.pivot_table(index="path", columns="ext", values=["header_offset", "compress_size", "compress_type"],
                          aggfunc="first")
    piv.columns = [f"{a}_{b}" for a, b in piv.columns]
    need = labels.merge(piv.dropna().reset_index(), on="path", how="inner")
    log.info("ECGs found in ZIP: %d / %d labelled", len(need), len(labels))
    need["lo"] = need[["header_offset_hea", "header_offset_dat"]].min(axis=1).astype("int64")
    need["hi"] = (need[["header_offset_hea", "header_offset_dat"]].max(axis=1) + 30 + 512 +
                  need[["compress_size_hea", "compress_size_dat"]].max(axis=1)).astype("int64")
    need["pack"] = (need["lo"] // int(cfg.zip_chunk_mb * 1e6)).astype(int)
    packs = sorted(need["pack"].unique())
    existing = set(store.list("shards"))
    done = {int(re.search(r"pack_(\d+)\.json", k).group(1)) for k in existing if re.search(r"pack_\d+\.json$", k)}
    tmp = Path(cfg.private_dir) / "pack_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    want = need.groupby("pack").size().to_dict()

    def _receipt_ok(p):                       # a pack from an older label set must be rebuilt
        loc = tmp / f"r{p}.json"
        return p, store.get(f"shards/pack_{p:05d}.json", loc) and json.loads(loc.read_text()).get("n") == want[p]

    with ThreadPoolExecutor(8) as ex:
        valid = {p for p, ok in ex.map(_receipt_ok, sorted(done & set(packs))) if ok}
    stale = len(done & set(packs)) - len(valid)
    todo = [p for p in packs if p not in valid]
    log.info("ZIP packs: %d total, %d already done, %d stale (rebuilt), %d to download (~%.0f MB each)",
             len(packs), len(valid), stale, len(todo), cfg.zip_chunk_mb)
    n_out = 10 * cfg.fs_out
    size = remote_size(url) if todo else 0
    t0, nbytes, lock = time.time(), [0], threading.Lock()

    def do_pack(p):
        g = need[need["pack"] == p]
        lo, hi = int(g["lo"].min()), int(g["hi"].max())
        hi = min(hi, size - 1)
        buf = fetch_range(_session(), url, lo, hi)
        X = {s: [] for s in ("train", "val", "test")}
        M = {s: [] for s in ("train", "val", "test")}
        for r in g.itertuples(index=False):
            try:
                hea = _member(buf, lo, int(r.header_offset_hea), int(r.compress_size_hea), int(r.compress_type_hea))
                dat = _member(buf, lo, int(r.header_offset_dat), int(r.compress_size_dat), int(r.compress_type_dat))
                x, fs = parse_wfdb(hea, dat)
                y = clean_resample(x, fs, cfg.fs_out, n_out)
            except Exception as e:
                log.debug("skip %s: %s", r.path, e)
                y = None
            if y is not None:
                X[r.split].append(y)
                M[r.split].append(r)
        keys = []
        d = tmp / f"p{p}"
        d.mkdir(exist_ok=True)
        for s in X:
            if not X[s]:
                continue
            key = f"shards/{s}/pack_{p:05d}"
            np.save(d / f"{s}.npy", np.stack(X[s]))
            meta = pd.DataFrame(M[s])[[c for c in labels.columns if c != "path"]]
            meta.to_parquet(d / f"{s}.parquet", index=False)
            store.put(d / f"{s}.npy", f"{key}.npy")
            store.put(d / f"{s}.parquet", f"{key}.parquet")
            keys.append([s, key, len(X[s])])
        (d / "r.json").write_text(json.dumps({"keys": keys, "n": len(g)}))
        store.put(d / "r.json", f"shards/pack_{p:05d}.json")             # receipt last = commit
        with lock:
            nbytes[0] += hi - lo
        return p, keys

    stop = threading.Event()

    def heartbeat():                          # a 256 MB pack takes minutes; show that bytes are flowing
        last, t_last = _LIVE[0], time.time()
        while not stop.wait(60):
            now, t = _LIVE[0], time.time()
            log.info("  downloading... %.0f MB/s now, %.1f GB received so far",
                     (now - last) / 1e6 / (t - t_last), now / 1e9)
            last, t_last = now, t

    if todo:
        log.info("Downloading %d packs, %d in parallel; first 'pack done' line comes after one full pack "
                 "(a few minutes), progress every 60 s", len(todo), cfg.zip_parallel)
        threading.Thread(target=heartbeat, daemon=True).start()
        with ThreadPoolExecutor(cfg.zip_parallel) as ex:
            futs = [ex.submit(do_pack, p) for p in todo]
            for i, f in enumerate(as_completed(futs), 1):
                p, keys = f.result()
                el = time.time() - t0
                mbps = nbytes[0] / 1e6 / max(el, 1e-6)
                left = (len(todo) - i) * cfg.zip_chunk_mb / max(mbps, 1e-6) / 60
                log.info("pack %d done (%d/%d) | %d ECGs | %.1f MB/s | ETA %.0f min", p, i, len(todo),
                         sum(k[2] for k in keys), mbps, left)
    stop.set()
    manifest = {"train": [], "val": [], "test": []}
    for p in packs:
        loc = tmp / f"r{p}.json"
        store.get(f"shards/pack_{p:05d}.json", loc)
        for s, key, n in json.loads(loc.read_text())["keys"]:
            manifest[s].append([key, n])
    man_loc.parent.mkdir(parents=True, exist_ok=True)
    man_loc.write_text(json.dumps(manifest))
    store.put(man_loc, man_key)
    return manifest


## 3 · Config
Change only this cell.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo")
from src.config import Config

cfg = Config(
    project_id="mimic-cxr-training",   # <-- your GCP project ID (same as the other projects)
    bucket="",                         # empty = "<project_id>-mimic-ckpt" (private)
    run_name="main",                   # same name = resume
    max_ecgs=None,                     # None = FULL dataset; e.g. 20000 for a quick trial (use another run_name)
    epochs=25, batch_size=256,
    zip_parallel=6,                    # parallel ZIP range downloads (~0.3 MB/s each from physionet.org)
)
# TIP: the download phase needs NO GPU. Run it with Accelerator = None (saves GPU quota); when the log says
# "shards ready", switch to GPU T4 x2 and Run All again -> it goes straight to training.

### Pre-flight check
Everything should say OK.

In [ ]:
# pre-flight: everything should say OK
from src.gcp import check_access
from src.waveforms import check_source
check_access(cfg.project_id, ["mimiciv_hosp.labevents"])
check_source(cfg)
import torch; print("  GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 4 · Load the pipeline
Imports the project code written above. The cells below are the steps of `src/pipeline.py::run()`, one per cell.

In [ ]:
import sys, time
from IPython.display import display
for _m in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]                       # always use the freshly written code (no stale imports)
import src.pipeline as _pipeline
globals().update({k: v for k, v in vars(_pipeline).items() if not k.startswith("__")})   # helpers + imports


class StopRun(Exception):
    """Raised on purpose (e.g. no GPU yet / paused): Run All again later to continue."""


def _stop(result):
    print(result)
    raise StopRun(result.get("status", "stopped") if isinstance(result, dict) else str(result))


fetcher = None
labels = None
mm = None


## 5 · Start
Logging, folders and the checkpoint store.

In [ ]:
t0 = time.time()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
out, priv = Path(cfg.work_dir), Path(cfg.private_dir)
for d in (out / "model", out / "tables", out / "figures", priv):
    d.mkdir(parents=True, exist_ok=True)
torch.manual_seed(cfg.seed)
store = make_store(cfg)

## 6 · Labels: K⁺ near each ECG
BigQuery, cached in the bucket; split by patient.

In [ ]:
lab = labels if labels is not None else lab_mod.build_labels(cfg, store)
log.info("Labels: %d ECGs / %d patients | split %s", len(lab), lab.subject_id.nunique(),
         lab.split.value_counts().to_dict())
display(lab.split.value_counts())

## 7 · Waveforms → shards (resumable)
Range requests on the PhysioNet zip; every finished pack is committed to the bucket.

In [ ]:
if fetcher is None and cfg.wave_source in ("zip", "auto"):
    from src import zipsource
    try:
        manifest = zipsource.build_packs(cfg, store, lab)
    except Exception as e:
        if cfg.wave_source == "zip":
            raise
        log.warning("ZIP route failed (%s) -> per-file downloads", e)
        manifest = waveforms.build_shards(cfg, store, lab)
else:
    manifest = waveforms.build_shards(cfg, store, lab, fetcher)
if cfg.train_needs_gpu and not torch.cuda.is_available():
    log.warning("All waveform shards are ready in the bucket, but this session has no GPU. "
                "Switch the accelerator to GPU T4 x2 and Run All again to train (shards are reused).")
    _stop({"status": "shards_ready_no_gpu", "n_shards": sum(len(v) for v in manifest.values())})
sets = {s: ShardSet(cfg, store, manifest[s]) for s in ("train", "val", "test")}
stats = label_stats(lab, cfg)

## 8 · Train ResNet-1D (resumable)
Full training state is saved every 10 minutes. The table shows every epoch, also the ones loaded from the checkpoint (★ = best).

In [ ]:
try:
    model, st = train(cfg, store, sets, stats)
except Paused as e:
    log.warning("PAUSED: %s -> Run All again (same run_name) to continue.", e)
    _stop({"status": "paused", "detail": str(e)})
display(history_table(st))

## 9 · Evaluate on test
Bootstrap CIs, subgroups, machine-measurement baseline, Grad-CAM beats.

In [ ]:
pred = predict(model, sets["test"], cfg, stats)
cls = evaluate.classification(pred, cfg)
reg = evaluate.regression(pred)
sub = evaluate.subgroups(pred, cfg)
try:
    mmdf = mm if mm is not None else lab_mod.machine_measurements(cfg, store)
    base = evaluate.baseline_machine(lab, mmdf, cfg)
except Exception as e:
    log.warning("machine-measurement baseline skipped: %s", e)
    base = {}
beats = evaluate.gradcam_beats(model, sets["test"], pred, cfg)
evaluate.plots(pred, cls, beats, st["history"], base, cfg, out / "figures")
display(cls)
display(reg)

## 10 · Save model + report
`.pth` bundle with everything inference needs.

In [ ]:
bundle = {"state_dict": model.state_dict(), "arch": "ResNet1D",
          "arch_kwargs": dict(width=cfg.width, kernel=cfg.kernel, dropout=cfg.dropout),
          "stats": stats, "fs": cfg.fs_out, "seconds": 10, "leads": waveforms.LEADS,
          "thresholds": dict(hyper=cfg.hyper_thr, severe=cfg.severe_thr, hypo=cfg.hypo_thr)}
torch.save(bundle, out / "model" / "ecg_k_resnet1d.pth")
cohort = dict(n_ecgs=int(len(lab)), n_patients=int(lab.subject_id.nunique()),
              by_split={s: int(n) for s, n in lab.split.value_counts().items()},
              n_waveforms_used={s: int(sets[s].n) for s in sets},
              hyper_prevalence=float((lab.k >= cfg.hyper_thr).mean()),
              hypo_prevalence=float((lab.k < cfg.hypo_thr).mean()),
              k_median=float(lab.k.median()), abs_dt_median_min=float(lab.k_dt_min.abs().median()))
metrics = dict(cohort=cohort, classification=cls.to_dict("records"), regression=reg,
               baseline_machine_measurements=base, training=dict(best_epoch=st["best_epoch"] + 1,
               best_val_auroc=st["best_auc"], epochs_run=len(st["history"])),
               gradcam={g: {k: v for k, v in b.items() if k.startswith(("cam_share", "n_beats"))}
                        for g, b in beats.items() if g != "t_ms"},
               config=cfg.to_dict(), runtime_min=round((time.time() - t0) / 60, 2))
(out / "metrics.json").write_text(json.dumps(metrics, indent=1, default=float))
(out / "model" / "model_card.json").write_text(json.dumps(
    {k: bundle[k] for k in ("arch", "arch_kwargs", "stats", "fs", "seconds", "leads", "thresholds")}
    | {"test": metrics["classification"], "regression": reg}, indent=1, default=float))
cls.to_csv(out / "tables" / "classification.csv", index=False)
sub.to_csv(out / "tables" / "subgroups.csv", index=False)
(out / "tables" / "training_history.json").write_text(json.dumps(st["history"], indent=1))
from src.report import build as build_report
build_report(out)

# patient-level predictions -> private only
(priv / "preds").mkdir(exist_ok=True)
pred[["study_id", "k", "pred_k", "p_hyper", "p_hypo"]].to_parquet(priv / "preds" / "test_predictions.parquet")
store.put_dir(priv / "preds", f"runs/{cfg.run_name}/private")
store.put_dir(out, f"runs/{cfg.run_name}/results")
if cfg.cleanup_private:
    shutil.rmtree(priv, ignore_errors=True)
a = cls.iloc[0]
log.info("DONE | test AUROC(%s)=%.3f (%.3f-%.3f) | MAE=%.3f | model -> %s", a.Target, a.AUROC, a["AUROC lo"],
         a["AUROC hi"], reg["mae"], store.uri(f"runs/{cfg.run_name}/results/model/ecg_k_resnet1d.pth"))
result = {"status": "complete", **metrics}
result

## 11 · Results report
Aggregate results only (safe for GitHub).

In [ ]:
# Report (aggregate only -> safe for GitHub) + model file
import re, shutil, glob
from IPython.display import Markdown, Image, display
rep = f"{cfg.work_dir}/report"
md = open(f"{rep}/REPORT.md", encoding="utf-8").read()
display(Markdown(re.sub(r"!\[.*?\]\(figures/.*?\)", "", md)))
for f in sorted(glob.glob(f"{rep}/figures/*.png")):
    display(Image(f))
shutil.make_archive("/kaggle/working/results_for_github", "zip", rep)
print("Results zip  : /kaggle/working/results_for_github.zip  -> unzip into the repo as results/")
print("Model (.pth) :", f"{cfg.work_dir}/model/ecg_k_resnet1d.pth", "(also in your bucket under runs/main/results/model/)")